In [8]:
from google import auth
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
import os

In [24]:
today = datetime.now()

## GCP CONNECTION

In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=7rUEe8irEk2VEyYCDgvdkN244G2jvd&access_type=offline&code_challenge=SxVt2SxehqR7lv7j_v8FQJ8qG3zbszXIvhy_ugdUHuM&code_challenge_method=S256

ERROR: There was a problem with web authentication. Try running again with --no-browser.
ERROR: (gcloud.auth.application-default.login) (missing_code) Missing code parameter in response.


In [24]:
df_flower = pd.read_csv("labels.csv")

In [25]:
df_flower

,id,Flower
0,0,pink primrose
1,1,hard-leaved pocket orchid
2,2,canterbury bells
3,3,sweet pea
4,4,english marigold
...,...,...
97,97,mexican petunia
98,98,bromelia
99,99,blanket flower
100,100,trumpet creeper


## DATABASE INFO

In [2]:
# initialize parameters
project_id = "AntherosFluffyDex"
region = "us-central1-c"
instance_name = "antheros-fluffy-dex"
INSTANCE_CONNECTION_NAME = "antherosfluffydex:us-central1:antheros-fluffy-dex" # i.e demo-project:us-central1:demo-instance
print(f"Your instance connection name is: {INSTANCE_CONNECTION_NAME}")
DB_USER = "sqlserver"

DB_PASS = "9R>%&-:ZXl&:FBZa"
DB_NAME = "Default"

Your instance connection name is: antherosfluffydex:us-central1:antheros-fluffy-dex


## DATABASE CONNECTION

In [23]:
from google.cloud.sql.connector import Connector
import sqlalchemy
from sqlalchemy import Column, Float, Integer, String, Table

# initialize Connector object
connector = Connector()

# function to return the database connection object
def getconn():
    conn = connector.connect(
        INSTANCE_CONNECTION_NAME,
        "pytds",
        user=DB_USER,
        password=DB_PASS,
        db=DB_NAME
    )
    return conn

# create connection pool with 'creator' argument to our connection object function
engine = sqlalchemy.create_engine(
    "mssql+pytds://",
    creator=getconn,
)

C:\Users\vtheu\AppData\Local\Temp\ipykernel_19852\1722016365.py:20: SADeprecationWarning: The dbapi() classmethod on dialect classes has been renamed to import_dbapi().  Implement an import_dbapi() classmethod directly on class <class 'sqlalchemy_pytds.dialect.MSDialect_pytds'> to remove this warning; the old .dbapi() classmethod may be maintained for backwards compatibility.
  engine = sqlalchemy.create_engine(


In [39]:
inspector = sqlalchemy.inspect(engine)

In [40]:
inspector.get_table_names()

['dim_flower', 'fact_history_predicts']

## TABLE CREATION IWTH SQL COMMAND

In [38]:
with engine.connect() as connection:
    # The transaction ensures the table creation is atomic
    with connection.begin() as transaction:
        connection.execute(text("""
            CREATE TABLE fact_history_predicts (
                id INTEGER PRIMARY KEY,
                flower_id INTEGER,
                date DATETIME
            )
        """))
    print("Table 'fact_history_predicts' created successfully.")

Table 'fact_history_predicts' created successfully.


In [5]:
with engine.connect() as connection:
    # The transaction ensures the table creation is atomic
    with connection.begin() as transaction:
        connection.execute(text("""
            ALTER TABLE dbo.dim_flower ADD image_name VARCHAR(255)
        """))
    print("success")

success


In [14]:
for file in os.listdir("download"):
    flower_name = file.split('.')[0].replace('_', " ")
    with engine.connect() as connection:
        with connection.begin():
            connection.execute(
                text("""
                    UPDATE dbo.dim_flower
                    SET image_name = :file
                    WHERE Flower = :flower_name
                """),
                {"file": file, "flower_name": flower_name}
            )

In [21]:
with engine.connect() as connection:
    with connection.begin():
        connection.execute(
            text("""
                UPDATE dbo.dim_flower
                SET image_name = :file
                WHERE Flower = :flower_name
            """),
            {"file": 'rose.jpg', "flower_name": 'rose'}
        )

In [31]:
with engine.connect() as connection:
    with connection.begin():
        connection.execute(
            text("""
CREATE TABLE fact_history_predicts (
    id INT IDENTITY(1,1) PRIMARY KEY,
    flower_id INTEGER,
    date DATETIME NOT NULL
);
            """),
           
        )

## INSERT TO

In [33]:
insert_query = text("""
    INSERT INTO fact_history_predicts (flower_id, date) 
    VALUES ( :flower_id, :date_val)
""")
params = { 
    "flower_id": 2, 
    "date_val": today  # Pass the raw datetime object here
}
with engine.connect() as connection:
    # The transaction ensures the table creation is atomic
    with connection.begin() as transaction:
        connection.execute(insert_query,params)

## TABLE UDATE WITH PANDAS

In [27]:
df_flower.to_sql(
    name = 'dim_flower',
    con = engine,
    schema = 'dbo',
    if_exists = 'replace',
    index = False
)

102

In [34]:
pd.read_sql("SELECT  * from dbo.fact_history_predicts", engine)

,id,flower_id,date
0,1,2,2025-08-14 14:35:25.340


In [35]:
result

,id,Flower
0,0,pink primrose


## BUCKET CHECK

In [13]:

from google.cloud import storage

try:
    # The client automatically finds the credentials via the environment variable
    storage_client = storage.Client()
    
    print("Authentication successful using service account!")
    print("Listing buckets:")
    
    for bucket in storage_client.list_buckets():
        print(f"- {bucket.name}")

except Exception as e:
    print(f"Authentication failed: {e}")
    print("\nPlease ensure the path to your service account key is correct and it has permissions.")

Authentication successful using service account!
Listing buckets:
- antherosfluffydex_cloudbuild
- flower_images_fluffy
